# 02 · Attribute, reference and source contracts

**OncoPlate v3.0.0 — implementation of the accepted v3.0 plan**

Run after the data audit. The proposed schema is not automatically domain-approved.

This is research software, not a validated cancer-risk or chemical-detection product. Run cells in order. Missing independently collected data or approval records are genuine prerequisites, not permission to substitute synthetic results.

In [ ]:
from pathlib import Path
import os, sys, json
ON_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
if ON_COLAB and not Path("/content/drive/MyDrive").exists():
    from google.colab import drive
    drive.mount("/content/drive")
ROOT = Path(os.environ.get("ONCOPLATE_DRIVE_ROOT", "/content/drive/MyDrive/OncoPlate_Research"))
pointer = ROOT / ".oncoplate_install.json"
preferred = json.loads(pointer.read_text())["repository_path"] if pointer.exists() else str(ROOT / "oncoplate-research")
REPO = Path(os.environ.get("ONCOPLATE_REPO", preferred))
if not (REPO / "src/oncoplate").exists():
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src/oncoplate").exists(): REPO = candidate; break
assert (REPO / "src/oncoplate").exists(), "Run the supplied installer notebook or set ONCOPLATE_REPO to the extracted repository."
sys.path.insert(0, str(REPO / "src"))
from oncoplate.config import load_config, paths, initialize
from oncoplate.io import read_json, write_json, read_table, write_table, read_jsonl, write_jsonl, utcnow
cfg = load_config(REPO, root=ROOT, mode=os.environ.get("ONCOPLATE_MODE", "research"))
p = paths(cfg)
print("Dataset:", cfg["study"]["dataset"], "| Mode:", cfg["mode"], "| Persistent root:", cfg["root"])


## 1. Inspect source-data completeness

In [ ]:
import pandas as pd
records=read_table(p['prepared']/"records.csv");ann=read_table(p['prepared']/"annotations.csv")
fields=[f for f in cfg['study']['independent_fields'] if f in ann]
summary=pd.DataFrame([{'field':f,'recorded_rows':int(ann[f].astype(str).str.len().gt(0).sum()),'vocabulary_size':ann.loc[ann[f].ne(''),f].nunique()} for f in fields])
display(summary);write_table(p['reports']/"field_completeness.csv",summary)

## 2. Read the executable contracts
Factual and evidential labels remain separate. Product/meal focal items need real visual binding.

In [ ]:
print((REPO/'docs/DATA_CONTRACTS.md').read_text())
print((REPO/'schemas/claim_ratings.schema.json').read_text())

## 3. Write a target-definition decision record without inventing approval

In [ ]:
definition={'version':'v3.0-implementation','target':'expert annotation endorsement for public foundation; independently adjudicated support for new benchmark',
'ordinal_exposure_tiers':False,'source_fields_are_not_reference_outcomes':True,
'item_pairing':'Preserve tuples from actual annotation rows. Never form a food-method cross product.',
'missing_labels':'Mask missing bundle fields. Recorded panel non-endorsement is not verified physical absence.',
'vocabulary':'Fit partition only after notebook 03','domain_review_status':'pending'}
write_json(p['prepared']/"target_specification_draft.json",definition)
print(json.dumps(definition,indent=2))

## 4. Run source and reference integrity tests

In [ ]:
import subprocess
subprocess.run([sys.executable,'-m','pytest','-q',str(REPO/'tests/test_claims_and_gates.py'),str(REPO/'tests/test_data_integrity.py')],cwd=REPO,check=True)

## Completion and resumption
Outputs are written to the displayed persistent dataset-specific directories. Keep raw inputs, reviewer records, arrays and checkpoints private. Re-running a completed fit verifies its configuration rather than silently changing it. Use notebook 99 only after reviewing aggregate results; nothing here pushes to GitHub automatically.
